# Sessão 6 · Prática guiada: explorar as vendas de 2024 e 2025

A direção da Morabeza ainda não quer um painel: quer respostas rápidas sobre dois anos de vendas.
Vais responder-lhes passo a passo, com NumPy e pandas.

**Como trabalhar**

1. Corre as células por ordem, de cima para baixo (Shift + Enter).
2. Onde vires `...`, substitui pelo teu código.
3. Depois de cada resposta há uma célula `confere(...)`. Diz-te se está certo, sem mostrar a solução.

Se uma célula der erro, lê a última linha da mensagem: é quase sempre aí que está a pista.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# Procura a pasta com os CSV: no Codespace (data/tabelas) ou ao lado do notebook (Colab)
candidatas = [Path("data/tabelas"), Path("../data/tabelas"), Path(".")]
PASTA = next(p for p in candidatas if (p / "vendas.csv").exists())
print("Dados em:", PASTA.resolve())

In [ ]:
import hashlib

def confere(valor, codigo):
    """Diz se a tua resposta está certa, sem mostrar a solução."""
    if valor is ...:
        print("Ainda não respondeste a este passo.")
        return
    if hasattr(valor, "sum"):   # um array ou uma coluna: confere a soma
        valor = valor.sum()
    try:
        texto = str(round(float(valor), 1))
    except (TypeError, ValueError):
        texto = str(valor)
    certo = hashlib.md5(texto.encode()).hexdigest()[:6] == codigo
    print("Certo!" if certo else "Ainda não. Revê o passo e volta a correr a célula.")

## Parte 1 · NumPy (10 min)

Quatro vendas pequenas, para aquecer. Primeiro com arrays, sem pandas.

In [ ]:
q = np.array([5, 25, 12, 40])          # quantidades
preco = np.array([100, 50, 1000, 90])  # preço unitário
q, preco

In [ ]:
totais = totais = q * preco
totais

In [ ]:
confere(totais, "9a5db8")

**1.2** Cria a máscara das vendas com quantidade igual ou superior a 20 e conta quantas são.

In [ ]:
grandes = q >= 20
n_grandes = grandes.sum()
grandes, n_grandes

In [ ]:
confere(n_grandes, "d1bd83")

**1.3** Com `np.where`, aplica 5% de desconto às vendas grandes (quantidade de 20 ou mais).
Guarda em `com_desconto` o total de cada venda já com desconto.

In [ ]:
com_desconto = np.where(grandes, totais * 0.95, totais)
com_desconto

In [ ]:
confere(com_desconto, "8a4f25")

## Parte 2 · Ler e conhecer os dados (10 min)

Agora as tabelas a sério. A célula seguinte lê as três tabelas. Repara no `parse_dates`.

In [ ]:
vendas = pd.read_csv(PASTA / "vendas.csv", parse_dates=["data"])
clientes = pd.read_csv(PASTA / "clientes.csv", parse_dates=["data_registo"])
produtos = pd.read_csv(PASTA / "produtos.csv")
vendas.head()

**2.1** Usa `info()` para ver as colunas e os tipos das vendas. Que tipo tem a coluna `data`? Há valores em falta?

In [ ]:
vendas.info()

**2.2** Quantas linhas tem a tabela de vendas? (usa `shape`)

In [ ]:
n_vendas = vendas.shape[0]
n_vendas

In [ ]:
confere(n_vendas, "85b6dd")

**2.3** Usa `describe()` nas vendas. Qual é a quantidade média por venda? E o preço máximo?

In [ ]:
vendas.describe()

**2.4** Quantos clientes há em Santiago? Usa `value_counts()` na coluna `ilha` dos clientes.

In [ ]:
clientes["ilha"].value_counts()

In [ ]:
n_santiago = clientes["ilha"].value_counts()["Santiago"]

In [ ]:
confere(n_santiago, "aba3bc")

## Parte 3 · Uma coluna nova (5 min)

A tabela não tem o total de cada venda. Cria-o, vetorizado, como fizeste com os arrays.

**3.1** Cria a coluna `total` (quantidade vezes preço unitário) e calcula a faturação dos dois anos.

In [ ]:
vendas["total"] = vendas["quantidade"] * vendas["preco_unitario"]
faturacao = vendas['total'].sum()
faturacao

In [ ]:
confere(faturacao, "c522a0")

## Parte 4 · Filtrar (10 min)

Lembra-te: com pandas usa-se `&` e `|`, e cada condição vai entre parênteses.

**4.1** Quantas vendas têm quantidade igual ou superior a 20?

In [ ]:
n_grandes = len(vendas[vendas["quantidade"]>=20])
n_grandes

In [ ]:
confere(n_grandes, "6cee6d")

**4.2** Quanto se faturou em 2025? Pista: `vendas["data"].dt.year` dá o ano de cada venda.

In [ ]:
fat_2025 = vendas.loc[vendas["data"].dt.year == 2025, "total"].sum()
fat_2025

In [ ]:
confere(fat_2025, "400ab7")

**4.3** Quantos clientes do tipo HORECA há no Sal? Usa duas condições com `&`.

In [ ]:
n_horeca_sal = len(clientes[(clientes["tipo"]=="HORECA") & (clientes["ilha"]=="Sal")])
n_horeca_sal

In [ ]:
confere(n_horeca_sal, "e23763")

**4.4** Quantos clientes são do Barlavento? Para estes dados, conta São Vicente, Sal, Santo Antão e Boa Vista. Usa `isin`.

In [ ]:
barlavento = ["São Vicente", "Sal", "Santo Antão", "Boa Vista"]
n_barlavento = clientes["ilha"].isin(barlavento).sum()
n_barlavento

In [ ]:
confere(n_barlavento, "ad3b03")

## Parte 5 · Agrupar (10 min)

`groupby` divide as linhas por grupo, faz uma conta a cada grupo e junta os resultados.

**5.1** Faturação por ano. Compara 2024 com 2025: a empresa cresceu?

In [ ]:
por_ano = vendas.groupby(vendas["data"].dt.year)["total"].sum()
por_ano

Quanto se faturou em 2024? Tira o valor de `por_ano` com `.loc`.

In [ ]:
fat_2024 = vendas.loc[vendas["data"].dt.year == 2024, "total"].sum()
fat_2024

In [ ]:
confere(fat_2024, "7e7559")

**5.2** Os 5 produtos que mais faturaram. Agrupa por `produto_id`, soma o `total` e ordena do maior para o menor.

In [ ]:
top5 = vendas.groupby("produto_id")["total"].sum().sort_values(ascending=False).head(5)
top5

Qual é o código do produto que mais faturou? É o primeiro rótulo do índice de `top5`.

In [ ]:
primeiro = top5.index[0]
primeiro

In [ ]:
confere(primeiro, "3e2f46")

**5.3** Quantos clientes há de cada tipo em cada ilha? Agrupa os clientes por `ilha` e `tipo` e usa `size()`.

In [ ]:
clientes.groupby(["ilha", "tipo"]).size()

## Parte 6 · Desafio (5 min)

A direção pondera dar 5% de desconto em todas as vendas com quantidade de 20 ou mais.
Quanto custaria esse desconto nos dois anos? Usa `np.where` sobre as colunas do DataFrame.

In [ ]:
valor_venda = vendas["quantidade"] * vendas["preco_unitario"]

vendas["total_desconto"] = np.where(
    vendas["quantidade"] >= 20,
    valor_venda * 0.05,
    0.0
)

custo = vendas["total_desconto"].sum()
custo

In [ ]:
confere(custo, "000797")

## Se acabares cedo

**E.1** Qual foi o mês com maior faturação? Pista: `vendas["data"].dt.to_period("M")` dá o mês de cada venda, e `idxmax()` devolve o rótulo do maior valor.

In [ ]:
faturacao_mensal = (
    vendas.assign(mes=vendas["data"].dt.to_period("M"))
    .groupby("mes")["faturacao"]  # ou vendas["quantidade"] * vendas["preco_unitario"]
    .sum()
)

melhor_mes = faturacao_mensal.idxmax()
melhor_mes

In [ ]:
confere(melhor_mes, "34c644")

**E.2** Os códigos do top 5 dizem pouco à direção. Mostra os nomes dos produtos.
Pista: `produtos.set_index("produto_id")` e depois `.loc[top5.index, "nome"]`.

In [ ]:
nomes_top5 = produtos.set_index("produto_id").loc[top5.index, "nome"]
nomes_top5

Na próxima sessão juntas estas tabelas com `merge`, mudas a forma com `pivot_table` e montas o painel de vendas no Laboratório 4.